# Homework For Class 4: RAG with LangChain

## 1. Filtered retrieval across two areas
Build a retriever that searches only policies/ and answer:

“What is the per-person limit for reimbursing a client meal, and how many days of annual leave do I get?”

Then remove the filter and run the same question again. Print the filenames of the retrieved chunks in both cases and explain why the results differ.

Files involved: expense_policy.pdf , employee_handbook.pdf
Topics: metadata filtering, area field, similarity_search with filter=

In [20]:
# Load kb folder
from ai_eng.settings import KNOWLEDGE_DIR
from ai_eng.loaders import load_corpus

docs = load_corpus(KNOWLEDGE_DIR)

No languages specified, defaulting to English.


Loaded 134 documents from D:\coding shuttle\ai engineer\practice-lab\ai-eng\src\ai_eng\4_langchain_rag\knowledge


In [21]:
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
from pydantic import SecretStr
import uuid

load_dotenv()

qdrant_client = QdrantClient("http://localhost:6333")

embed_client = OpenAIEmbeddings(
    base_url="https://openrouter.ai/api/v1",
    model="nvidia/nemotron-3-embed-1b:free",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
    check_embedding_ctx_length=False
)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

if not qdrant_client.collection_exists("helpdesk_kb"):
    qdrant_client.create_collection(
        collection_name="helpdesk_kb",
        vectors_config=models.VectorParams(size=2048, distance=models.Distance.COSINE)
    )

vector_store = QdrantVectorStore(client=qdrant_client, embedding=embed_client,collection_name="helpdesk_kb")
added_ids = vector_store.add_documents(
    documents=chunks,
    ids=[str(uuid.uuid4()) for _ in chunks]
)

print(f"Added {len(added_ids)} points to Qdrant")

print(f"Added {len(added_ids)} points to Qdrant")
print(vector_store.collection_name)



Added 191 points to Qdrant
Added 191 points to Qdrant
helpdesk_kb


In [22]:
policies_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.area",
            match=models.MatchValue(value="policies")
        )
    ]
)

similarity_retriever = vector_store.as_retriever(
    search_type = "similarity",
    kwargs={"k":4},
    filter = policies_filter
)

result1 = similarity_retriever.invoke("What is the per-person limit for reimbursing a client meal, and how many days of annual leave do I get")
print("with filter\n")
for doc in result1:
    print(doc.metadata["filename"], doc.metadata.get("page"))

result2 = vector_store.similarity_search(
    "What is the per-person limit for reimbursing a client meal, and how many days of annual leave do I get",
    k=4
)
print("without filter\n")
for doc in result2:
    print(doc.metadata["filename"], doc.metadata.get("page"))

with filter

expense_policy.pdf 1
expense_policy.pdf 4
expense_policy.pdf 1
employee_handbook.pdf 1
without filter

expense_policy.pdf 1
expense_policy.pdf 4
expense_policy.pdf 1
employee_handbook.pdf 1


## 2. MMR vs plain similarity
The onboarding wiki covers VPN setup, two-factor auth, and the password
manager in separate sections that are semantically close.

Ask: “How do I keep my accounts secure?”

Run the retriever twice — once with search_type="similarity" and once with
search_type="mmr" — both at k=4 . 

Print the h2 heading from each retrieved chunk’s metadata. Explain in a comment which set of results is more useful and why.

Files involved: onboarding.md
Topics: MarkdownHeaderTextSplitter metadata, MMR diversity, retriever search_type

In [23]:
sim_retriever = vector_store.as_retriever(
    search_type="similarity",
    kwargs={"k":4},
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    kwargs={"k":4, "lambda_mult":0.0},
)

sim_results = sim_retriever.invoke("How do I keep my accounts secure?")
print("similarity results")
for doc in sim_results:
    h2_header = doc.metadata.get("h2","No h2 header found")
    print(f"{h2_header}")

print("\nmmr results")
mmr_result = mmr_retriever.invoke("How do I keep my accounts secure?")
for doc in sim_results:
    h2_header = doc.metadata.get("h2","No h2 header found")
    print(f"{h2_header}")


similarity results
No h2 header found
No h2 header found
No h2 header found
No h2 header found

mmr results
No h2 header found
No h2 header found
No h2 header found
No h2 header found


## 3. Streaming answer with live source list
Write a function stream_answer(question: str) that:
1. Retrieves chunks with the existing retriever
2. Prints each source filename on its own line before streaming starts
Homework For Class 4: RAG with LangChain 1
3. Streams the answer token-by-token using rag_chain.stream()
Call it with: “What happens if I miss the on-call pager alert?”
Files involved: onboarding.md
Topics: .stream() , sources from retriever, combining explicit and LCEL steps

In [27]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",     # any model id from openrouter.ai/models
    temperature=0,                   # 0 keeps answers focused and repeatable
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Zomato's internal help-desk assistant.
Answer using only the context below.

Rules:
- If the answer is not in the context, say you don't know.
- Cite the sources you used as [1], [2], matching the numbers in the context.
- Never invent policies, amounts or procedures.

Context:
{context}""",
    ),
    ("human", "{question}"),
])

def cite(doc) -> str:
    m = doc.metadata
    if "page" in m:
        return f"{m['filename']}, page {m['page'] + 1}"           # PDF: page is zero-based
    section = m.get("h3") or m.get("h2") or m.get("h1")
    if section:
        return f"{m['filename']}, section '{section}'"            # Markdown: heading path
    return m["filename"]                                          # CSV and others


def format_docs(docs) -> str:
    """Number each chunk and label it with its origin so the LLM can cite it."""
    return "\n\n---\n\n".join(
        f"[{i}] ({cite(d)})\n{d.page_content}" for i, d in enumerate(docs, start=1)
    )

rag_chain = (
    {
        "context":sim_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

def stream_answer(question: str):
    print(f"Question: {question}")
    retrieved_docs = sim_retriever.invoke(question)

    for doc in retrieved_docs:
        filename = doc.metadata.get("filename","Unknown Source")
        print(f"- {filename}")

    for chunk in rag_chain.stream(question):
        print(chunk, end="", flush=True)
    print()

stream_answer("What happens if I miss the on-call pager alert?")


Question: What happens if I miss the on-call pager alert?
- onboarding.md
- payments_runbook.md
- payments_runbook.md
- payments_runbook.md
The context states that on-call engineers must "acknowledge alerts within 10 minutes" [1], but it does not specify what happens if an alert is missed.


## 4. Swap the embedding provider
Replace OpenAIEmbeddings with HuggingFaceEmbeddings ( sentence-transformers/all-MiniLM-L6-
v2 , 384 dims) and rebuild the in-memory vector store from the same chunks .
Ask the same question from Part 4 (“How many vacation days do I get?”) with
both stores and print:
the top result’s filename and score from each
the vector dimension of each model
Write one sentence noting any difference in the retrieved chunks.
Files involved: full corpus
Topics: embedding provider swap, dimension mismatch consequence,
similarity_search_with_score


In [ ]:
# Problem 4: Swap the embedding provider (Free Local Model)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 1. Initialize the free HuggingFace embedding model
embeddings_hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. Rebuild the in-memory vector store from the same chunks using HuggingFace
vector_store_hf = InMemoryVectorStore.from_documents(chunks, embeddings_hf)

# 3. Define the question from Part 4
question = "How many vacation days do I get?"

# 4. Query the store using similarity_search_with_score
results_hf = vector_store_hf.similarity_search_with_score(question, k=1)

# 5. Print the required metrics for HuggingFace store
print("--- HuggingFace Embeddings Store ---")
if results_hf:
    doc, score = results_hf[0]
    print(f"Top Result Filename: {doc.metadata.get('filename')}")
    print(f"Similarity Score: {score}")

# Inspect vector dimension directly from the embedding model instance
hf_dim = len(embeddings_hf.embed_query("test"))
print(f"Vector Dimension: {hf_dim}")


# --- Comparison with previous store (assuming vector_store & embeddings were OpenAI) ---
# Uncomment below if you have your original OpenAI vector store loaded to compare side-by-side:
"""
results_openai = vector_store.similarity_search_with_score(question, k=1)
print("\n--- OpenAI Embeddings Store ---")
if results_openai:
    doc_oa, score_oa = results_openai[0]
    print(f"Top Result Filename: {doc_oa.metadata.get('filename')}")
    print(f"Similarity Score: {score_oa}")

openai_dim = len(embeddings.embed_query("test")) # or 1536 for text-embedding-3-small
print(f"Vector Dimension: {openai_dim}")
"""

"""
Difference Commentary / Note:
The HuggingFace model (all-MiniLM-L6-v2) outputs a vector dimension of 384, whereas OpenAI models 
typically output 1536 dimensions, meaning you cannot mix embeddings from different providers in the same index 
due to a dimension mismatch error. 
"""

## 5. Question the support FAQ
The support_faq.csv has 63 rows loaded as one Document per row. Build a
separate in-memory vector store from only those 63 chunks (no other files) and
answer:
“My order was cancelled — when will I get my money back?”
Print the top 3 results with their category metadata. Then change k from 3 to 1 and
note whether the single best result is enough to answer the question fully.
Files involved: support_faq.csv
Topics: CSVLoader with metadata_columns , isolated vector store, choosing k

In [28]:
# Problem 5: Question the support FAQ

from langchain_core.vectorstores import InMemoryVectorStore

# 1. Filter the already loaded corpus to isolate only support_faq.csv chunks
faq_chunks = [doc for doc in chunks if doc.metadata.get("filename") == "support_faq.csv"]

print(f"Loaded {len(faq_chunks)} chunks from support_faq.csv")

# 2. Build a separate in-memory vector store using only these FAQ chunks
faq_vector_store = InMemoryVectorStore.from_documents(faq_chunks, embed_client)

question = "My order was cancelled — when will I get my money back?"

# 3. Query with k=3 and print top results with category metadata
print("\n--- Results for k = 3 ---")
results_k3 = faq_vector_store.similarity_search_with_score(question, k=3)

for i, (doc, score) in enumerate(results_k3, 1):
    category = doc.metadata.get("category", "No category")
    print(f"Result {i}:")
    print(f"  - Category: {category}")
    print(f"  - Score: {score}")
    print(f"  - Content: {doc.page_content[:150]}...\n")

# 4. Query with k=1
print("--- Results for k = 1 ---")
results_k1 = faq_vector_store.similarity_search_with_score(question, k=1)

if results_k1:
    doc_k1, score_k1 = results_k1[0]
    print(f"Top Result Category: {doc_k1.metadata.get('category')}")
    print(f"Content: {doc_k1.page_content}")

"""
Note on sufficiency (k=3 vs k=1):
A single best result (k=1) is generally enough to answer the question fully if the FAQ row 
contains the exact timeline and refund policy. However, keeping k=3 is useful to ensure 
that edge cases (like partial refunds, store credits vs. original payment method, or 
processing exceptions) aren't missed by relying on just one matching row.
"""

Loaded 63 chunks from support_faq.csv


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1792108800000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.'}}}